# OUTCOME MACE: all baseline, at least 1-100 genes

Ordinal 3-class target (0/1/2), scored with a label-weighted quadratic
kappa. Class probabilities are collapsed to a continuous ordinal score
(`proba @ [0, 1, 2]`), and two cut points are tuned on each validation
fold to maximise weighted QWK.

Feature construction mirrors the severity notebook: top-20 SNPs raw, the
rest as PCA components plus a sum, on top of the full clinical panel.

## Setup

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from cardi_utils import (
    load_data,
    get_snp_cols,
    mace_weights,
    make_stratified_folds,
    train_one_fold_lgbm,
    sample_params_multiclass,
    quadratic_weighted_kappa,
    apply_thresholds_ordinal,
    optimize_thresholds_for_weighted_qwk,
    rank_snps_by_shap_mace_lgbm,
)

pd.set_option("display.max_columns", None)

train, test = load_data()
snp_cols = get_snp_cols(train)

## Features

In [2]:
fixed_cols_mace = [
    "Age_Baseline", "Age_Diag", "BMI", "BSA", "Genre",
    "Epaiss_max", "Gradient", "TVNS", "FEVG", "ATCD_MS", "SYNCOPE", "Diam_OG",
    "Variant.Pathogene"
]

X = train[fixed_cols_mace + snp_cols].copy()
y = train["OUTCOME MACE"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Model: random parameter search with threshold optimisation

In [3]:
def fit_lgbm_mace_qwk_cv(
    train: pd.DataFrame,
    target_col: str,
    base_cols: list[str],
    snp_cols: list[str],
    stability_df: pd.DataFrame | None = None,
    selected_snp_top_n: int = 20,
    candidate_pool_size: int | None = None,
    use_pca: bool = True,
    n_pcs: int = 10,
    add_snp_sum: bool = True,
    test_size: float = 0.2,
    split_seed: int = 42,
    mace_label_weights: dict[int, float] | None = None,
    use_sample_weights: bool = True,  # affects TRAINING ONLY
    n_splits_cv: int = 5,
    n_param_samples: int = 40,
    num_boost_round: int = 4000,
    early_stopping_rounds: int = 100,
    verbose_cv: bool = False,
    threshold_grid: int = 40,  # for threshold optimization inside CV
):

    if mace_label_weights is None:
        mace_label_weights = {0: 1.0, 1: 3.0, 2: 4.0}

    y_all = train[target_col].astype(int)
    tr_idx, te_idx = train_test_split(
        train.index, test_size=test_size, stratify=y_all, random_state=split_seed
    )
    y_tr = y_all.loc[tr_idx].values
    y_te = y_all.loc[te_idx].values

    classes = np.sort(np.unique(y_tr))
    if not np.array_equal(classes, np.array([0, 1, 2])):
        raise ValueError(f"{target_col} must have classes [0,1,2], got {classes.tolist()}")

    snp_cols = [c for c in snp_cols if c in train.columns]
    if not snp_cols:
        raise ValueError("No SNP columns found in train.")

    if stability_df is not None and "snp" in stability_df.columns:
        ranked = [s for s in stability_df["snp"] if s in snp_cols]
        if not ranked:
            ranked = snp_cols.copy()
    else:
        ranked = snp_cols.copy()

    selected = ranked[:selected_snp_top_n]
    remaining = [s for s in ranked if s not in selected]
    candidate_pool = remaining if candidate_pool_size is None else remaining[:candidate_pool_size]

    if use_pca and len(candidate_pool) < max(n_pcs, 2):
        raise ValueError("Not enough SNPs in candidate_pool for PCA.")
    
    fixed_cols = base_cols + selected

    X_fixed_tr = train.loc[tr_idx, fixed_cols].copy()
    X_fixed_te = train.loc[te_idx, fixed_cols].copy()

    X_snp_tr = train.loc[tr_idx, candidate_pool].copy()
    X_snp_te = train.loc[te_idx, candidate_pool].copy()

    X_tr_df = X_fixed_tr.copy()
    X_te_df = X_fixed_te.copy()

    pca = None
    pc_cols: list[str] = []

    if use_pca:
        pca = PCA(n_components=n_pcs, random_state=split_seed)
        pcs_tr = pca.fit_transform(X_snp_tr.values)
        pcs_te = pca.transform(X_snp_te.values)
        pc_cols = [f"SNP_PC{i+1}" for i in range(n_pcs)]
        X_tr_df = pd.concat([X_tr_df, pd.DataFrame(pcs_tr, index=tr_idx, columns=pc_cols)], axis=1)
        X_te_df = pd.concat([X_te_df, pd.DataFrame(pcs_te, index=te_idx, columns=pc_cols)], axis=1)

    if add_snp_sum:
        X_tr_df["SNP_SUM"] = X_snp_tr.sum(axis=1).values
        X_te_df["SNP_SUM"] = X_snp_te.sum(axis=1).values

    feature_cols = list(X_tr_df.columns)

    w_tr_eval = np.array([mace_label_weights[int(y)] for y in y_tr], dtype=float)
    w_te_eval = np.array([mace_label_weights[int(y)] for y in y_te], dtype=float)

    w_tr_fit = w_tr_eval if use_sample_weights else None
    w_te_fit = w_te_eval if use_sample_weights else None

    folds = make_stratified_folds(y_tr, n_splits=n_splits_cv, seed=split_seed)
    rng = np.random.default_rng(split_seed)

    best_qwk = -np.inf
    best_params: dict | None = None
    best_num_boost: int | None = None
    best_thresholds: tuple[float, float] | None = None

    X_tr_np = X_tr_df.values
    class_vals = np.array([0.0, 1.0, 2.0], dtype=float)

    for _ in range(n_param_samples):
        params = sample_params_multiclass(rng=rng, num_class=3)

        fold_scores: list[float] = []
        fold_best_iters: list[int] = []
        fold_t1s: list[float] = []
        fold_t2s: list[float] = []

        for fold in folds:
            tr_i, va_i = fold.train_idx, fold.valid_idx

            booster, it, proba_va = train_one_fold_lgbm(
                params=params,
                X_tr=X_tr_np[tr_i],
                y_tr=y_tr[tr_i],
                X_va=X_tr_np[va_i],
                y_va=y_tr[va_i],
                feature_cols=feature_cols,
                num_boost_round=num_boost_round,
                early_stopping_rounds=early_stopping_rounds,
                verbose=verbose_cv,
                w_tr=None if w_tr_fit is None else w_tr_fit[tr_i],
                w_va=None if w_tr_fit is None else w_tr_fit[va_i],
            )

            # Convert class-probabilities -> continuous ordinal score
            score_va = proba_va @ class_vals

            # Optimize thresholds on the validation fold to maximize weighted QWK
            t1, t2, qwk_va = optimize_thresholds_for_weighted_qwk(
                y_true=y_tr[va_i],
                score=score_va,
                sample_weight=w_tr_eval[va_i],
                n_grid=40,
            )

            fold_scores.append(float(qwk_va))
            fold_best_iters.append(int(it))
            fold_t1s.append(float(t1))
            fold_t2s.append(float(t2))

        mean_qwk = float(np.mean(fold_scores))
        mean_it = int(np.round(np.mean(fold_best_iters)))
        mean_t1 = float(np.mean(fold_t1s))
        mean_t2 = float(np.mean(fold_t2s))

        if mean_qwk > best_qwk:
            best_qwk = mean_qwk
            best_params = params
            best_num_boost = max(1, mean_it)
            best_thresholds = (mean_t1, mean_t2)

    if best_params is None or best_num_boost is None or best_thresholds is None:
        raise RuntimeError("MACE CV search failed.")

    dtrain_full = lgb.Dataset(
        X_tr_df.values, label=y_tr, weight=w_tr_fit, feature_name=feature_cols, free_raw_data=True
    )
    dtest = lgb.Dataset(
        X_te_df.values, label=y_te, weight=w_te_fit, feature_name=feature_cols, reference=dtrain_full, free_raw_data=True
    )

    final_model = lgb.train(
        params=best_params,
        train_set=dtrain_full,
        num_boost_round=best_num_boost,
        valid_sets=[dtest],
        valid_names=["test"],
        callbacks=[
            lgb.early_stopping(early_stopping_rounds, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )
    best_it_final = int(final_model.best_iteration or best_num_boost)

    proba_tr = final_model.predict(X_tr_df.values, num_iteration=best_it_final)
    proba_te = final_model.predict(X_te_df.values, num_iteration=best_it_final)

    score_tr = proba_tr @ class_vals
    score_te = proba_te @ class_vals

    t1, t2 = best_thresholds
    yhat_tr = apply_thresholds_ordinal(score_tr, t1, t2)
    yhat_te = apply_thresholds_ordinal(score_te, t1, t2)

    qwk_tr = quadratic_weighted_kappa(y_tr, yhat_tr, sample_weight=w_tr_eval)
    qwk_te = quadratic_weighted_kappa(y_te, yhat_te, sample_weight=w_te_eval)

    metrics = {
        "cv_best_qwk_weighted": float(best_qwk),
        "train_qwk_weighted": float(qwk_tr),
        "test_qwk_weighted": float(qwk_te),
        "best_threshold_t1": float(t1),
        "best_threshold_t2": float(t2),
        "best_num_boost_round": int(best_num_boost),
        "best_iteration_final": int(best_it_final),
        "used_sample_weights_for_training": bool(use_sample_weights),
    }

    return best_params, final_model, pca, metrics, candidate_pool, feature_cols, selected, best_thresholds

## Rank SNPs by weighted mean |SHAP|

In [4]:
mace_rank_df = rank_snps_by_shap_mace_lgbm(
    train=train,
    target_col="OUTCOME MACE",
    base_cols=fixed_cols_mace,
    snp_cols=snp_cols,
    use_weights_for_training=True,
    use_weights_for_shap=True,
)

mace_rank_df

,snp,mean_abs_shap,rank
0,SNP278,0.201140,1
1,SNP117,0.095859,2
2,SNP137,0.084014,3
3,SNP259,0.064614,4
4,SNP205,0.062893,5
...,...,...,...
283,SNP230,0.000000,284
284,SNP228,0.000000,285
285,SNP233,0.000000,286
286,SNP229,0.000000,287


## Fit

In [5]:
best_params_mace, model_mace, pca_mace, metrics_mace, pool_mace, feat_mace, trusted_mace, (t1, t2) = fit_lgbm_mace_qwk_cv(
    train=train,
    target_col="OUTCOME MACE",
    base_cols=fixed_cols_mace,
    snp_cols=snp_cols,
    stability_df=mace_rank_df,
    mace_label_weights=mace_weights,
    use_sample_weights=True,
)
print(metrics_mace)

{'cv_best_qwk_weighted': 0.45303757378010195, 'train_qwk_weighted': 0.6164918177402334, 'test_qwk_weighted': 0.29791140693137474, 'best_threshold_t1': 0.6926466558789389, 'best_threshold_t2': 0.9044883281153124, 'best_num_boost_round': 70, 'best_iteration_final': 40, 'used_sample_weights_for_training': True}


In [6]:
#model_mace.save_model('mace_model_sub5.txt')

## Predict on test

In [7]:
ID_COL = "trustii_id"
N_PCS = 10
pc_cols = [f"SNP_PC{i+1}" for i in range(N_PCS)]

X_test = test[fixed_cols_mace + trusted_mace + pool_mace].copy()
pcs = pca_mace.transform(X_test[pool_mace].to_numpy())
for i, c in enumerate(pc_cols):
    X_test[c] = pcs[:, i]

X_test["SNP_SUM"] = X_test[pool_mace].sum(axis=1)
X_test.drop(columns=pool_mace, inplace=True)
X_test = X_test[feat_mace]

proba = model_mace.predict(
    X_test,
    num_iteration=getattr(model_mace, "best_iteration", None)  # uses best_iteration if present
)

if proba.ndim == 1:
    n_classes = 3  # for OUTCOME MACE {0,1,2}
    proba = proba.reshape(-1, n_classes)

# Convert probs -> predicted class {0,1,2}
pred_class = np.argmax(proba, axis=1).astype(int)

pred_df_mace = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    "OUTCOME MACE": pred_class
})

pred_df_mace

,trustii_id,OUTCOME MACE
0,1,0
1,2,2
2,3,0
3,4,0
4,5,0
...,...,...
144,145,0
145,146,0
146,147,2
147,148,0


Hand the predictions to `03_submission.ipynb`.

In [8]:
pred_df_mace.to_csv("pred_mace.csv", index=False)